In [1]:
import ollama
import pdfplumber
from pathlib import Path
import io
import os
import mistralai
from dotenv import load_dotenv
import json
import re
import time

In [2]:
from mistralai.client import Mistral

In [3]:
load_dotenv()
assert os.getenv("MISTRAL_API_KEY") is not None
client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

In [4]:
library_dir = Path('library2')

In [5]:
response = client.chat.complete(
    model="mistral-large-latest",
    messages=[
        {"role": "user", "content":  'Schreibe einen Satz auf Deutsch.'}
    ],
)

print(response.choices[0].message.content)

Hier ist ein Satz auf Deutsch:

**"Der Himmel leuchtet heute in einem wunderschönen Blau, während die Vögel fröhlich zwitschern."**

Möchtest du einen bestimmten Stil (formell, poetisch, umgangssprachlich etc.) oder ein Thema? 😊


# Gefährdungsszenarien

In [10]:
source_dir = library_dir / 'Gefährdungsszenarien'

## Alle .pdf als .md speichern:

In [15]:
for file_path in source_dir.iterdir():
    
    if not file_path.suffix.lower() == '.pdf':
        continue
    md_file_path_source = source_dir / file_path.with_suffix(".md").name
    
    if not md_file_path_source.exists():
        with open(file_path, "rb") as f:
            pdf_file = io.BytesIO(f.read())

        text = ""
        with pdfplumber.open(pdf_file) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        with open(md_file_path_source, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"Markdown-Datei gespeichert unter: {md_file_path_source}")

Markdown-Datei gespeichert unter: library2/Gefährdungsszenarien/38-Anschlag-C-Kampfstoff-GD-de.md
Markdown-Datei gespeichert unter: library2/Gefährdungsszenarien/26-Mangellage-Erdoelversorgung-GD-de.md


## Wichtige Konzepte aus den Szenarios extrahieren und in json-Format bringen:

In [16]:
def analyze_szenario_prompt(text: str) -> str:
    return f"""
Du bist ein Experte für Krisenmanagement und Szenario-Extraktion.
Du erhältst einen Text zu einem bestimmten Krisenszenario und sollst dazu folgende Fragen beantworten:

1. Um welche Gefahr handelt es sich bei dem Szenario? Antworte in maximal zwei Worten!
2. Wie lässt sich die Gefahr kurz in ein oder zwei Sätzen beschreiben?
3. Was für Personen können im Krisenfall zum Schutz der Bevölkerung beigezogen werden? Zähle einzeln die Rollen, Berufe oder Funktinen auf, die sie innehaben.
4. Was für Hilfsmittel und Werkzeuge stehen zur Verfügung, die im Krisenfall eingesetzt werden können?
5. Was für Massnahmen kommen in Frage? Nenne nur konkrete Massnahmen. "Unterstützung der Betroffenen" ist zum Beispiel nicht konkret geung.
6. Was für rechtliche Grundlagen werden genannt? Für jeden rechtlichen Text, nenne sowohl den vollen Namen wie auch die Abkürzung.
7. Läuft dieses Szenario typischerweise in bestimmten Phasen ab? Wenn ja, nenne diese Phasen der Reihe nach. Wenn nein, antworte mit "Nicht vorhanden."
8. Gibt es eine Skala von Schwerestufen für die Gefahr? Wenn ja, nenne diese Schwerestufen der Reihe nach. Wenn nein, antworte mit "Nicht vorhanden."
9. Was sind die möglichen Auswirkungen und Schäden eines solchen Szenarios?
10. Was für Einflussfaktoren wirken auf Entwicklung und Schwere der Gefahr?
11. Zähle die konkreten Beispiele solcher Gefahrereignisse auf, die im Text genannt werden. Nenne zu jedem Ort, Datum und eine kurze Zusammenfassung des Verlaufs.

Beantworte die Fragen in der Reihenfolge wie sie gestellt werden.
Antworte ausschliesslich auf Deutsch.
Formatiere den Text als markdown.
Verwende keine verschachtelten Listen. Beschränke dich in deinen Antworten auf eine einfache Bullet Point-Liste ohne Unterkategorien.

Hier ist der Text:
{text}
"""

In [17]:
def make_json_prompt(text: str) -> str:
    return f"""
    Kannst du den unten angegebenen Text zu einem JSON-Objekt machen?

    Das Format muss genau so sein:

    {{
        "Gefahr": "...",
        "Beschreibung": "...",
        "Rollen": ["...", "..."],
        "Hilfsmittel": ["...", "..."],
        "Massnahmen": ["...", "..."],
        "Rechtsgrundlagen": ["...", "..."],
        "Phasen": ["...", "..."],
        "Schwerestufen": ["...", "..."],
        "Auswirkungen": ["...", "..."],
        "Einflussfaktoren": ["...", "..."],
        "Ereignisbeispiele": ["...", "..."]
    }}

    Gib nur das JSON-Objekt aus, ohne zusätzliche Erklärungen, und ohne Markdown-Codeblöcke (z. B. ```json).
    Achte darauf, dass das JSON valide ist und direkt gespeichert werden kann.

    Hier ist der Text: {text}
    """

In [18]:
json_szenarios = []
for file_path in source_dir.iterdir():

    print(file_path)
    
    if not file_path.suffix.lower() == '.md':
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    prompt_analyze = analyze_szenario_prompt(text)

    response_analyze = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "user", "content":  prompt_analyze}
    ],
    ).choices[0].message.content

    print(f"Response plain text: {response_analyze[:100]}")

    prompt_json = make_json_prompt(response_analyze)

    response_json = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "user", "content":  prompt_json}
    ],
    ).choices[0].message.content

    json_szenario = json.loads(response_json)
    json_szenario['Quelle'] =  str(source_dir / file_path.with_suffix(".pdf").name)

    print('JSON Szenario:')
    print(json_szenario.keys())

    json_szenarios.append(json_szenario)


library2/Gefährdungsszenarien/01-Hagelschlag-GD-de.pdf
library2/Gefährdungsszenarien/02-Starkregen-GD-de.md
Response plain text: ```markdown
- **1. Gefahr**
  Starkregen

- **2. Beschreibung der Gefahr**
  Starkniederschlag mit h
JSON Szenario:
dict_keys(['Gefahr', 'Beschreibung', 'Rollen', 'Hilfsmittel', 'Massnahmen', 'Rechtsgrundlagen', 'Phasen', 'Schwerestufen', 'Auswirkungen', 'Einflussfaktoren', 'Ereignisbeispiele', 'Quelle'])
library2/Gefährdungsszenarien/38-Anschlag-C-Kampfstoff-GD-de.md
Response plain text: ```markdown
- **1. Gefahr**
  Chemischer Anschlag

- **2. Beschreibung der Gefahr**
  Ein gewalttäti
JSON Szenario:
dict_keys(['Gefahr', 'Beschreibung', 'Rollen', 'Hilfsmittel', 'Massnahmen', 'Rechtsgrundlagen', 'Phasen', 'Schwerestufen', 'Auswirkungen', 'Einflussfaktoren', 'Ereignisbeispiele', 'Quelle'])
library2/Gefährdungsszenarien/38-Anschlag-C-Kampfstoff-GD-de.pdf
library2/Gefährdungsszenarien/01-Hagelschlag-GD-de.md
Response plain text: ```markdown
- **1. Gefahr**
  Ha

In [19]:
json_szenarios[0]

{'Gefahr': 'Starkregen',
 'Beschreibung': 'Starkniederschlag mit hoher Intensität, der auf Böden trifft, die das Wasser nicht aufnehmen können. Dies führt zu Oberflächenabfluss, der Erosions- und Überflutungsschäden verursacht.',
 'Rollen': ['Feuerwehrleute',
  'Rettungssanitäter',
  'Polizei',
  'Technisches Hilfspersonal (z.B. für Räumungsarbeiten)',
  'Fachpersonal (z.B. Dachdecker)',
  'Einsatzkräfte des Bevölkerungsschutzes',
  'Mitarbeiter von Versorgungsunternehmen (Strom, Wasser, Telekommunikation)',
  'Psychologische Betreuer',
  'Mitglieder von Zivilschutzorganisationen',
  'Fachkräfte für Abwasser- und Entwässerungssysteme'],
 'Hilfsmittel': ['Entwässerungspumpen',
  'Räumungsfahrzeuge und -geräte',
  'Notstromaggregate',
  'Sandsäcke',
  'Warn- und Alarmsysteme',
  'Drohnen für Lageerkundung',
  'Schlauch- und Pumpsysteme',
  'Kommunikationsgeräte (Funk, Satellitentelefone)',
  'Schutzausrüstung (z.B. wasserdichte Kleidung, Helme)',
  'Notunterkünfte und Zelte'],
 'Massnahm

## Konzeptdeails extrahieren

In [20]:
index_keys = [
    "Rollen",
    "Hilfsmittel",
    "Massnahmen",
    "Rechtsgrundlagen",
    "Phasen",
    "Schwerestufen",
    "Auswirkungen",
    "Einflussfaktoren",
    "Ereignisbeispiele"
]

def make_detail_prompt(text: str, item: str) -> str:
    return f"""
    Was sagt der folgende Text über {item}?

    Beantworte ausschliesslich die Frage. Bleibe kurz und knapp, aber auch präzise.

    Hier ist der Text: {text}
    """

In [21]:
md_content_all = "# Gefährdungsszenarien\n\n"
all_szenarios = {}
for szenario in json_szenarios:
    gefahr = szenario['Gefahr']
    print(gefahr)
    gefahr = gefahr.replace(' ', '_')
    gefahr_file = library_dir / Path(gefahr + '_index.md')
    all_szenarios[gefahr] = gefahr_file

    
    md_content_all += f"- [{szenario['Gefahr']}]({ Path(gefahr + '_index.md')})\n"

    md_content_szenario = f"# {szenario['Gefahr']}\n\n"
    source_file = Path(szenario['Quelle']).parent / Path(szenario['Quelle']).with_suffix(".md").name

    with open(source_file, "r", encoding="utf-8") as f:
        source_text = f.read()

    for key in index_keys:

        print(key)

        output_path_detail = str(library_dir / Path(f"{gefahr}_{key}.md"))

        md_content_szenario += f"- [{key}]({output_path_detail.split('/')[-1]})\n"
        response_collection = []

        if Path(output_path_detail).exists():
            print(f"{output_path_detail} existiert bereits")
            continue
        
        for item in szenario[key]:

            print(item)
    
            detail_prompt = make_detail_prompt(source_text, item)
    
            response_detail = client.chat.complete(
                model="mistral-large-latest",
                messages=[
                    {"role": "user", "content":  detail_prompt}
            ],
            ).choices[0].message.content
            time.sleep(30) 

            response_detail = f"## {item}\n\n{response_detail}"
            print(response_detail)
            response_collection.append(response_detail)

        combined_text = f" # {key}\n\n {"\n\n".join(response_collection)}"

        with open(output_path_detail, "w", encoding="utf-8") as f:
            f.write(f"Quelle:[{szenario['Quelle']}]({szenario['Quelle']})\n\n{combined_text}")

        print(f"{output_path_detail} erstellt.")

    print(md_content_szenario, gefahr_file)
    with open(gefahr_file, "w", encoding="utf-8") as f:
        f.write(md_content_szenario)

output_path = library_dir / Path("Gefährdungsszenarien_index.md")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(md_content_all)   

Starkregen
Rollen
Feuerwehrleute
## Feuerwehrleute

Der Text beschreibt Feuerwehrleute als **zentrale Einsatzkräfte** bei Starkregen mit Oberflächenabfluss:

- Sie retten eingeschlossene Personen (z. B. aus Kellern, Fahrzeugen).
- Sie pumpen überflutete Keller und Tiefgaragen aus.
- Sie räumen Geröll, Schlamm und Hindernisse von Straßen und Infrastrukturen.
- Sie unterstützen bei der Wiederherstellung der Entwässerung und provisorischen Reparaturen (z. B. Dächer).
- Sie arbeiten unter Extrembedingungen (Überlastung, Übermüdung, persönliche Betroffenheit).
- Sie sind Teil der überregionalen Hilfe bei großflächigen Ereignissen.
Rettungssanitäter
## Rettungssanitäter

Der Text beschreibt Rettungssanitäter als **Einsatzkräfte**, die bei Starkregen mit Oberflächenabfluss:

- **Personen aus Gefahrensituationen retten** (z. B. aus überfluteten Gebäuden, Fahrzeugen oder Unterführungen).
- **Verletzte versorgen** (z. B. bei Ertrinkungsgefahr, Quetschungen oder psychischer Belastung).
- **Durch 

# Test Szenario-Generation

In [22]:
output_path

PosixPath('library2/Gefährdungsszenarien_index.md')

In [23]:
with open(output_path, "r", encoding="utf-8") as f:
    index_file = f.read()

In [24]:
index_file

'# Gefährdungsszenarien\n\n- [Starkregen](Starkregen_index.md)\n- [Chemischer Anschlag](Chemischer_Anschlag_index.md)\n- [Hagelschlag](Hagelschlag_index.md)\n- [Erdölmangellage](Erdölmangellage_index.md)\n- [Starker Schneefall](Starker_Schneefall_index.md)\n'

In [88]:
keyword = 'Starker Schneefall'

In [89]:
def get_file_path(keyword, markdown_text):
    pattern = rf'\[{keyword}\]\((.*?)\)'
    match = re.search(pattern, markdown_text)
    return match.group(1) if match else None

In [90]:
gefahr_path = get_file_path(keyword, index_file)
print(gefahr_path)  # Ausgabe: Starkregen_index.md

Starker_Schneefall_index.md


In [91]:
with open(library_dir / gefahr_path, "r", encoding="utf-8") as f:
    gefahr_index_file = f.read()

In [92]:
relevant_text = []
for item in ['Phasen', 'Schwerestufen', 'Einflussfaktoren']:
    file = get_file_path(item, gefahr_index_file)
    with open(library_dir / file, "r", encoding="utf-8") as f:
        t = f.read()
    relevant_text.append(t)

In [93]:
relevant_text = '\n\n'.join(relevant_text)

In [94]:
len(relevant_text)

11682

In [122]:
def make_szenario_verlauf_prompt(text: str, gefahr: str) -> str:
    return f"""
    Im folgenden Text wirst du wichtige Informationen finden über Phasen, Schwerestufen und Einflussfaktoren eines Gefährdungsszenarios.
    Unsere Aufgabe ist die Konzipierung von verschiedenen Szenarien für eine Übung für den Ernstfall.

    Wir benötigen ein Szenario mit {gefahr} mit vier Verlaufsphasen: Vorphase, Hauptphase tiefere Eskalationsstufe, Hauptphase höhere Eskalationsstufe, Nachphase. 

    Entwerfe ein Szenario unter Inbezugahme der vergübaren Informationen.
    Liste die Phasen in iherer Reihenfolge auf und beschreibe die Merkmale von jeder Phase.

    Konzentriere dich ausschliesslich auf den Tatsachenverlauf. Formuliere keine Ziele, und auch keine Handlungsempfehlungen.
    Beschreibe auch keine Reaktionen oder getroffenen Massnahmen. Verzichte auch auf die Nennung von mittelbaren oder Langzeitfolgen, die erst nach der Nachphase eintreten.
    Das Szenario soll so verlaufen, wie wenn niemand eingreift.
    Es geht hier darum, den Rahmen der Grundbedingungen zu legen.

    Arbeite ausschliesslich mit den Informationen, die im Text zu finden sind!
    
    Hier ist der Text: {text}
    """

In [123]:
szenario_verlauf_prompt = make_szenario_verlauf_prompt(relevant_text, keyword)

In [124]:
response = client.chat.complete(
    model="mistral-large-latest",
    messages=[
        {"role": "user", "content":  szenario_verlauf_prompt}
    ],
)

print(response.choices[0].message.content)

### **Szenario: Starker Schneefall mit vier Verlaufsphasen**

#### **1. Vorphase (ca. 48 Stunden vor Ereignisbeginn)**
- **Meteorologische Lage**:
  - Ein Tiefdruckgebiet über Westeuropa lenkt feuchte, maritime Luftmassen in Richtung Mittelland.
  - Eine stationäre Front bildet sich aus, die über mehrere Tage hinweg intensive Niederschläge bringt.
  - Temperaturen sinken auf Werte knapp über oder unter dem Gefrierpunkt.
- **Schneelage**:
  - Bereits vorhandene Schneedecke von **30 cm** im Mittelland.
  - Boden ist gefroren, was die Haftung von Neuschnee begünstigt.
- **Wettervorhersage**:
  - Warnungen vor **starken Schneefällen** (50–70 cm in 72 Stunden) werden ausgegeben.
  - Begleitende Windböen (bis 60 km/h) werden prognostiziert.
- **Zeitpunkt**:
  - Beginn der Vorphase an einem **Freitagmorgen**, mit erwartetem Schneefall ab **Sonntagabend**.
  - Hohe Verkehrsdichte durch Wochenend- und Ferienverkehr (z. B. Adventszeit).

---

#### **2. Hauptphase – tiefere Eskalationsstufe (Tag 

In [125]:
with open('szenario-entwurf_7.md', "w", encoding="utf-8") as f:
    f.write(response.choices[0].message.content)   

In [126]:
def make_describe_prompt(text: str, gefahr: str) -> str:
    return f"""
    Der folgende Text handelt von einem fiktiven Gefährdungsszenario zum Thema {gefahr}.
    Welche Aspekte des Szenarioverlaufs werden in dem Beispiel beleuchtet?

    Antworte in einer Liste von kurzen, abstrakten Stichworten.
    
    Hier ist der Text: {text}
    """

In [127]:
describe_prompt = make_describe_prompt(response.choices[0].message.content, keyword)

In [129]:
response = client.chat.complete(
    model="mistral-large-latest",
    messages=[
        {"role": "user", "content":  describe_prompt}
    ],
)

print(response.choices[0].message.content)

Hier eine Liste der abstrakten Stichworte zu den beleuchteten Aspekten des Szenarioverlaufs:

- **Zeitliche Phasierung** (Vor-, Haupt-, Nachphase)
- **Meteorologische Entwicklung** (Tiefdruck, Fronten, Temperaturverlauf)
- **Niederschlagsintensität** (Schneemengen, Dauer, Art des Schnees)
- **Windwirkung** (Böen, Verwehungen)
- **Vorbelastung** (vorhandene Schneedecke, Bodenbeschaffenheit)
- **Frühwarnung** (Prognosen, Warnstufen, Vorlaufzeit)
- **Saisonaler Kontext** (Verkehrsaufkommen, Ferienzeit)
- **Infrastrukturausfälle** (Verkehr, Energie, Kommunikation)
- **Verkehrsbehinderungen** (Straßen, Schienen, Luftfahrt)
- **Versorgungsengpässe** (Lebensmittel, Medikamente, Heizmaterial)
- **Gebäudeschäden** (Dacheinstürze, Schneelast, Baumschäden)
- **Krisenmanagement** (Einsatzkräfte, Priorisierung, externe Hilfe)
- **Räumungs- und Wiederherstellungsprozesse** (Zeitbedarf, Prioritäten)
- **Regionale Betroffenheit** (Stadt vs. Land, Dichte der Infrastruktur)
- **Sekundärgefahren** (Dachl

In [139]:
def make_describe_prompt(text: str, gefahr: str) -> str:
    return f"""
    Der folgende Text handelt von einem fiktiven Gefährdungsszenario zum Thema {gefahr}.

    Wandle den Text in JSON-Format um. Für jede zeitliche Phase ein key, und die Beschreibung der entsprechenden Phase als dazugehöriger value.
    Es soll also so viele keys wie zeitliche Phasen geben. Verschiedene Exkalationsstufen in der Hauptphase gelten als separate Phasen.
    Du kannst dich in der Regel gut an der Titelstruktur orientieren.

    Verzichte auf Verschachtelungen.
    
    Hier ist der Text: {text}
    """

In [140]:
describe_prompt = make_describe_prompt(response.choices[0].message.content, keyword)

In [141]:
response = client.chat.complete(
    model="mistral-large-latest",
    messages=[
        {"role": "user", "content":  describe_prompt}
    ],
)

print(response.choices[0].message.content)

Hier ist die gewünschte JSON-Umwandlung ohne Verschachtelungen, bei der jede zeitliche Phase (inkl. Eskalationsstufen) als separater Key mit einer Beschreibung als Value dargestellt wird:

```json
{
  "Vorphase": "Vorbereitungs- und Frühwarnphase vor dem eigentlichen Schneefallereignis. Meteorologische Analysen führen zur Ausgabe von Warnstufen (gelb, orange, rot). Behörden aktivieren Notfallpläne, stellen Räumfahrzeuge und Streusalz bereit. Die Bevölkerung wird informiert und bereitet sich vor (z. B. Vorräte, Schutzmaßnahmen). Vorlaufzeit: Stunden bis Tage zwischen Warnung und Ereignisbeginn.",

  "Hauptphase_Stufe1_Intensiver_Schneefall": "Beginn des starken Schneefalls mit ersten Auswirkungen: Mehrere Zentimeter Schnee pro Stunde, leichte bis mäßige Windböen und beginnende Schneeverwehungen. Erste Verkehrsbehinderungen und Stromausfälle in exponierten Gebieten. Einsatzkräfte beginnen mit Räumungen kritischer Bereiche. Bevölkerung wird aufgefordert, unnötige Fahrten zu vermeiden und 

# Hinzufügen von (fiktiven) Örtlichkeiten

In [117]:
orte = [
    "Tourismusgebiet in einer Alpenregion",
    "Eine grössere Stadt in der Schweiz",
    "Das Areal von einem Open Air-Musikfestival in einem ländlichen Gebiet",
    "Ein grosses Stadion während eines Fussballspiels",
    "Die ganze Schweiz",
    "Die ganze Schweizer Alpenregion.",
    "Das Schweizer Mittelland",
    "Davos während dem WEF"
]